In [ ]:
# Setup seed for reproducibility
import random
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("PyTorch version:", torch.__version__)


Torch: 2.5.1+cu121
CUDA available: True
GPU: NVIDIA GeForce RTX 3060


In [24]:
# Load and prepare dataset
import pandas as pd

CSV_PATH = "../qnn_dataset/qnn_dataset_v2/qnn_features_v2.csv"

df = pd.read_csv(CSV_PATH)
print("Dataset shape:", df.shape)
print("Label distribution:\n", df["label"].value_counts())

# Filter out truncated AST samples if present
if "ast_truncated" in df.columns:
    df = df[df["ast_truncated"] == 0].copy()

# Extract feature columns
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
assert "label" in num_cols, "Label column not found"

feature_cols = [c for c in num_cols if c != "label"]

# Remove constant columns
if "parse_ok" in feature_cols and df["parse_ok"].nunique() == 1:
    feature_cols.remove("parse_ok")

X = df[feature_cols].astype(np.float32).values
y = df["label"].astype(np.float32).values


print(f"Features: {X.shape[1]}, Samples: {X.shape[0]}")

Shape: (1578, 142)
Label distribution:
 label
0    1169
1     409
Name: count, dtype: int64
n_features = 135


In [ ]:
# Split dataset: 70% train, 15% val, 15% test
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp
)

# Standardize features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train).astype(np.float32)
X_val = scaler.transform(X_val).astype(np.float32)
X_test = scaler.transform(X_test).astype(np.float32)

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

Train/Val/Test: (1103, 135) (236, 135) (237, 135)


In [ ]:
# Create PyTorch datasets and dataloaders
from torch.utils.data import Dataset, DataLoader

class QNNDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X)
        self.y = torch.from_numpy(y).view(-1, 1)

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

batch_size = 32

train_loader = DataLoader(
    QNNDataset(X_train, y_train), 
    batch_size=batch_size, 
    shuffle=True
)
val_loader = DataLoader(
    QNNDataset(X_val, y_val), 
    batch_size=batch_size, 
    shuffle=False

))

test_loader = DataLoader(    shuffle=False

    QNNDataset(X_test, y_test),     batch_size=batch_size, 

In [27]:
# Check PennyLane installation
import pennylane as qml
print("PennyLane version:", qml.__version__)


PennyLane version: 0.44.0
Name: pennylane
Version: 0.44.0
Summary: PennyLane is a cross-platform Python library for quantum computing, quantum machine learning, and quantum chemistry. Train a quantum computer the same way as a neural network.
Home-page: 
Author: 
License: 
Location: c:\Users\giakh\.conda\envs\NCKH\Lib\site-packages
Platform info:           Windows-10-10.0.26200-SP0
Python version:          3.11.14
Numpy version:           2.4.1
Scipy version:           1.17.0
JAX version:             None
Installed devices:
- default.clifford (pennylane-0.44.0)
- default.gaussian (pennylane-0.44.0)
- default.mixed (pennylane-0.44.0)
- default.qubit (pennylane-0.44.0)
- default.qutrit (pennylane-0.44.0)
- default.qutrit.mixed (pennylane-0.44.0)
- default.tensor (pennylane-0.44.0)
- null.qubit (pennylane-0.44.0)
- reference.qubit (pennylane-0.44.0)
- lightning.qubit (pennylane_lightning-0.44.0)


In [ ]:
# Build Hybrid Quantum Neural Network
import torch.nn as nn

# Quantum circuit parameters
n_features = X_train.shape[1]
n_qubits = 6
n_q_layers = 3

# Create quantum device (CPU-based)
pl_dev = qml.device("lightning.qubit", wires=n_qubits)
print(f"Quantum device: {pl_dev.name} with {n_qubits} qubits")

@qml.qnode(pl_dev, interface="torch", diff_method="adjoint")
def quantum_circuit(inputs, weights):
    """Quantum circuit with angle embedding and entangling layers"""
    qml.AngleEmbedding(inputs, wires=range(n_qubits), rotation="Y")
    qml.StronglyEntanglingLayers(weights, wires=range(n_qubits))
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

weight_shapes = {"weights": (n_q_layers, n_qubits, 3)}
qlayer = qml.qnn.TorchLayer(quantum_circuit, weight_shapes)

class HybridQNN(nn.Module):
    """Hybrid classical-quantum neural network"""
    def __init__(self, n_features, n_qubits):
        super().__init__()
        # Classical pre-processing
        self.pre = nn.Sequential(
            nn.Linear(n_features, 64),
            nn.ReLU(),
            nn.Linear(64, n_qubits),
            nn.Tanh()
        )
        # Quantum layer
        self.q = qlayer
        # Classical post-processing
        self.post = nn.Sequential(
            nn.Linear(n_qubits, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )

    def forward(self, x):
        x = self.pre(x)
        x = self.q(x)
        x = self.post(x)
        return x


# Initialize model on CPU (quantum device requires CPU)print(f"Total parameters: {sum(p.numel() for p in model.parameters())}")

torch_device = torch.device("cpu")print(f"Model device: {torch_device}")
model = HybridQNN(n_features=n_features, n_qubits=n_qubits).to(torch_device)

Using PennyLane device: lightning.qubit
Torch device: cpu


In [ ]:
# Setup loss function and optimizer
# Handle class imbalance with weighted loss
n_pos = float((y_train == 1).sum())
n_neg = float((y_train == 0).sum())
pos_weight = torch.tensor([n_neg / n_pos], device=torch_device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

print(f"Class balance - Positive: {int(n_pos)}, Negative: {int(n_neg)}")
print(f"Positive weight: {pos_weight.item():.4f}")


pos_weight: 2.8566434383392334


In [ ]:
# Training loop with early stopping
from sklearn.metrics import (
    precision_recall_fscore_support, 
    roc_auc_score, 
    accuracy_score
)

def evaluate(model, loader):
    """Evaluate model on a dataset"""
    model.eval()
    all_logits, all_y = [], []
    
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(torch_device)
            yb = yb.to(torch_device)
            logits = model(xb)
            all_logits.append(logits.cpu())
            all_y.append(yb.cpu())
    
    logits = torch.cat(all_logits).numpy().reshape(-1)
    y_true = torch.cat(all_y).numpy().reshape(-1)
    probs = 1.0 / (1.0 + np.exp(-logits))
    y_pred = (probs >= 0.5).astype(np.int32)

    acc = accuracy_score(y_true, y_pred)
    p, r, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", zero_division=0
    )
    
    try:
        auc = roc_auc_score(y_true, probs)
    except:
        auc = float("nan")
    
    return {"acc": acc, "precision": p, "recall": r, "f1": f1, "auc": auc}

# Training configuration
max_epochs = 50
patience = 8
best_f1 = 0.0
patience_counter = 0

print("Starting training...\n")

for epoch in range(1, max_epochs + 1):
    # Training phase
    model.train()
    total_loss = 0.0
    
    for xb, yb in train_loader:
        xb = xb.to(torch_device)
        yb = yb.to(torch_device)

        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * xb.size(0)

    train_loss = total_loss / len(train_loader.dataset)
    val_metrics = evaluate(model, val_loader)

    print(f"Epoch {epoch:02d} | Loss: {train_loss:.4f} | "
          f"Val - Acc: {val_metrics['acc']:.4f} P: {val_metrics['precision']:.4f} "
          f"R: {val_metrics['recall']:.4f} F1: {val_metrics['f1']:.4f} "
          f"AUC: {val_metrics['auc']:.4f}")



    # Save best modelprint(f"\nTraining completed. Best validation F1: {best_f1:.4f}")

    if val_metrics["f1"] > best_f1:

        best_f1 = val_metrics["f1"]            break

        patience_counter = 0            print(f"\nEarly stopping at epoch {epoch}")

        torch.save({        if patience_counter >= patience:

            "model": model.state_dict(),        patience_counter += 1

            "scaler_mean": scaler.mean_,    else:

            "scaler_scale": scaler.scale_,        print(f"  → Model saved (F1: {best_f1:.4f})")

            "feature_cols": feature_cols        }, "best_qnn.pt")

Epoch 01 | train_loss=1.0188 | val_acc=0.8263 val_p=0.9167 val_r=0.3607 val_f1=0.5176 val_auc=0.8414
Epoch 02 | train_loss=0.9949 | val_acc=0.8220 val_p=0.6557 val_r=0.6557 val_f1=0.6557 val_auc=0.8630
Epoch 03 | train_loss=0.9567 | val_acc=0.7966 val_p=0.5867 val_r=0.7213 val_f1=0.6471 val_auc=0.8570
Epoch 04 | train_loss=0.8923 | val_acc=0.7881 val_p=0.5696 val_r=0.7377 val_f1=0.6429 val_auc=0.8383
Epoch 05 | train_loss=0.8182 | val_acc=0.7754 val_p=0.5526 val_r=0.6885 val_f1=0.6131 val_auc=0.8424
Epoch 06 | train_loss=0.7326 | val_acc=0.8093 val_p=0.6333 val_r=0.6230 val_f1=0.6281 val_auc=0.8397
Epoch 07 | train_loss=0.6684 | val_acc=0.7839 val_p=0.5735 val_r=0.6393 val_f1=0.6047 val_auc=0.8301
Epoch 08 | train_loss=0.5910 | val_acc=0.7797 val_p=0.5672 val_r=0.6230 val_f1=0.5938 val_auc=0.8378
Epoch 09 | train_loss=0.5191 | val_acc=0.7839 val_p=0.5758 val_r=0.6230 val_f1=0.5984 val_auc=0.8393
Epoch 10 | train_loss=0.4674 | val_acc=0.7754 val_p=0.5556 val_r=0.6557 val_f1=0.6015 val_a

In [ ]:
# Evaluate on test set
ckpt = torch.load("best_qnn.pt", map_location=torch_device)
model.load_state_dict(ckpt["model"])
model.eval()

test_metrics = evaluate(model, test_loader)

print("\n" + "="*50)

print("TEST SET RESULTS")

print("="*50)print("="*50)

for metric, value in test_metrics.items():    print(f"{metric.capitalize():>12}: {value:.4f}")

C:\Users\giakh\AppData\Local\Temp\ipykernel_10260\3720944894.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load("best_qnn.pt", map_location=torch_device)


TEST: {'acc': 0.8016877637130801, 'precision': 0.6666666666666666, 'recall': 0.45901639344262296, 'f1': 0.5436893203883495, 'auc': 0.8062593144560358}


In [ ]:
# Detailed classification report
from sklearn.metrics import confusion_matrix, classification_report

# Get predictions on test set
model.eval()
all_logits, all_y = [], []

with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(torch_device)
        logits = model(xb).cpu().numpy().reshape(-1)
        all_logits.append(logits)
        all_y.append(yb.numpy().reshape(-1))

logits = np.concatenate(all_logits)
y_true = np.concatenate(all_y)
probs = 1.0 / (1.0 + np.exp(-logits))
y_pred = (probs >= 0.5).astype(int)

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

print("\nDetailed Classification Report:")
print(classification_report(y_true, y_pred, digits=4, target_names=["Non-Vulnerable", "Vulnerable"]))

Confusion matrix:
 [[162  14]
 [ 33  28]]

Report:
               precision    recall  f1-score   support

         0.0     0.8308    0.9205    0.8733       176
         1.0     0.6667    0.4590    0.5437        61

    accuracy                         0.8017       237
   macro avg     0.7487    0.6897    0.7085       237
weighted avg     0.7885    0.8017    0.7885       237

